# Laboratorio Centinela — Fases 1, 2 y 3 en un solo cuaderno
**Profundización I: Deep Learning · Maestría en Ciencia de Datos · USTA**
*Tomás Rendón Bustamante* · *Juan Sebastián Gamba Aguirre*

Este cuaderno es una **herramienta para explorar y explicar** el proyecto durante la sustentación: se mueven deslizadores y se ve qué pasa. **No reemplaza** los cuadernos de las Fases 1–2 y 3 (ahí están los resultados oficiales). Cada sección dice qué se simplificó.

**Cómo usarlo (Google Colab):** 1) Menú *Entorno de ejecución → Ejecutar todo* (o ejecute las celdas en orden). 2) Acepte el permiso de Drive. 3) Mueva los deslizadores. Las secciones 1 y 2 no necesitan GPU; la 3 usa GPU si hay (*Entorno de ejecución → Cambiar tipo → T4*).

**Qué lee de su Drive** (carpeta `MyDrive/Centinela/`): la serie del IDEAM `0026177030-NIVEL_H.csv` (si no está, la celda 0.2 le deja subirla) y `Fase_3/centinela_recortes_sar_224.npz` (recortes de radar). No se necesitan claves de Copernicus ni de ningún servicio.

## 0. Preparación

In [ ]:
# 0.1 Librerías y carpeta del proyecto en Drive
import os, glob, time, warnings
import numpy as np, pandas as pd
import ipywidgets as widgets
from IPython.display import display
warnings.filterwarnings('ignore')
try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
except ImportError:
    get_ipython().system('pip -q install plotly')
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

try:
    from google.colab import drive, output
    output.enable_custom_widget_manager()      # necesario para que las gráficas se actualicen solas al mover los deslizadores
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/Centinela'
except ImportError:                            # fuera de Colab: use la variable de entorno CENTINELA_BASE
    BASE = os.environ.get('CENTINELA_BASE', os.path.expanduser('~/Centinela'))
print('Carpeta del proyecto:', BASE, '| existe:', os.path.isdir(BASE))

# Estilo común de todas las gráficas
AZUL, NARANJA, GRIS, VERDE = '#1f4e79', '#e07b00', '#8a8a8a', '#0b6f6f'
def estilo(fig, titulo='', alto=360, **kw):
    fig.update_layout(template='plotly_white', title=dict(text=titulo, x=0.01), height=alto, margin=dict(l=55, r=20, t=60, b=50),
                      font=dict(family='Arial', size=13), legend=dict(orientation='h', y=1.12), hovermode='closest', **kw)
    return fig
def vivo(widgets_lista, funcion, rapido=True):
    """Conecta los deslizadores: la gráfica se actualiza sola al moverlos."""
    for w in widgets_lista:
        if hasattr(w, 'continuous_update'): w.continuous_update = rapido
        w.observe(lambda cambio: funcion(), names='value')
    funcion()

In [ ]:
# 0.2 Localiza los archivos de datos (busca dentro de la carpeta del proyecto)
def buscar(nombre):
    hallados = glob.glob(os.path.join(BASE, '**', nombre), recursive=True)
    return hallados[0] if hallados else None

RUTA_IDEAM = buscar('0026177030-NIVEL_H.csv')
RUTA_NPZ = buscar('centinela_recortes_sar_224.npz')

if RUTA_IDEAM is None:
    try:                                   # en Colab: permite subir el CSV del IDEAM y lo guarda en Drive
        from google.colab import files
        print('No encontré el CSV del IDEAM. Seleccione 0026177030-NIVEL_H.csv')
        subido = files.upload()
        os.makedirs(BASE, exist_ok=True)
        RUTA_IDEAM = os.path.join(BASE, '0026177030-NIVEL_H.csv')
        with open(RUTA_IDEAM, 'wb') as f:
            f.write(list(subido.values())[0])
    except ImportError:
        pass

print('Serie del IDEAM  :', RUTA_IDEAM)
print('Recortes de radar:', RUTA_NPZ)
assert RUTA_IDEAM, 'Falta el CSV del IDEAM (0026177030-NIVEL_H.csv).'

## 1. Fase 1 — ¿Qué tan difícil es predecir el nivel del río?
Se usa la serie del IDEAM (solo registros **Definitivo**, 2020–2025), igual que en la Fase 1.

In [ ]:
# 1.1 Carga de la serie del IDEAM (solo definitivos 2020-2025)
raw = pd.read_csv(RUTA_IDEAM)
raw['fecha'] = pd.to_datetime(raw['fechaObservacion'])
d = raw[raw['nivelAprobacion'].astype(str).str.contains('Definitivo', case=False)]
d = d[(d['fecha'] >= '2020-01-01') & (d['fecha'] < '2026-01-01')].drop_duplicates('fecha')
serie = d.set_index('fecha')['valorObservado'].astype(float).sort_index()
print(f'Registros totales: {len(raw):,} | definitivos 2020-2025: {len(serie):,}'.replace(',', ' '))
print('Periodo:', serie.index.min(), '->', serie.index.max())

In [ ]:
# 1.2 Tabla de modelado: nivel actual, rezagos y objetivo a +h horas (unidos por fecha exacta)
REZAGOS = (12, 24, 48, 72)

def arma_tabla(h_horas):
    t = pd.Timedelta(hours=h_horas)
    df = pd.DataFrame({'nivel': serie})
    df['objetivo'] = serie.reindex(serie.index + t).values
    for r in REZAGOS:
        df[f'nivel_{r}h'] = serie.reindex(serie.index - pd.Timedelta(hours=r)).values
    df['fecha_objetivo'] = df.index + t
    df = df.dropna().sort_values('fecha_objetivo')
    df['cambio'] = df['objetivo'] - df['nivel']              # lo que queremos predecir (cm)
    df['cambio_previo'] = df['nivel'] - df['nivel_12h']      # cuánto subió en las 12 h anteriores
    return df

def particion(df):
    n = len(df); i1, i2 = int(0.70 * n), int(0.85 * n)       # cronológica 70/15/15
    return {'entrenamiento': df.iloc[:i1], 'validación': df.iloc[i1:i2], 'prueba': df.iloc[i2:], 'todo': df}

TABLAS = {h: arma_tabla(h) for h in (12, 24, 36)}
print({h: len(t) for h, t in TABLAS.items()})

In [ ]:
# 1.3 EXPLORADOR (se actualiza solo): ¿con qué frecuencia sube el río y qué tan buenas son las reglas simples?
def metricas(real, pred):
    vp = int(((real) & (pred)).sum()); fp = int((~real & pred).sum()); fn = int((real & ~pred).sum())
    prec = vp / (vp + fp) if vp + fp else 0.0; sens = vp / (vp + fn) if vp + fn else 0.0
    f1 = 2 * prec * sens / (prec + sens) if prec + sens else 0.0
    return vp, fp, fn, prec, sens, f1

w_h = widgets.Dropdown(options=[12, 24, 36], value=12, description='Horizonte (h)')
w_u = widgets.IntSlider(value=4, min=1, max=10, description='Umbral (cm)')
w_c = widgets.Dropdown(options=['entrenamiento', 'validación', 'prueba', 'todo'], value='prueba', description='Conjunto')
txt13 = widgets.HTML()
fig13 = go.FigureWidget(estilo(go.Figure(), 'Cambio de nivel y nivel del río', 330))
fig13.add_trace(go.Histogram(x=[0], nbinsx=60, marker_color=GRIS, name='cambio de nivel'))
fig13.update_xaxes(title='Cambio de nivel (cm)'); fig13.update_yaxes(title='casos')
fig13b = go.FigureWidget(estilo(go.Figure(), 'Nivel del río en el periodo elegido (cm)', 260))
fig13b.add_trace(go.Scattergl(x=[0], y=[0], mode='lines', line=dict(color=AZUL, width=1), name='nivel'))

def actualiza13():
    h, u, cj = w_h.value, w_u.value, w_c.value
    df = particion(TABLAS[h])[cj]
    real = (df['cambio'] > u).values
    reglas = {'«nunca sube»': np.zeros(len(df), bool),
              '«sube si ya subió > umbral en las 12 h previas»': (df['cambio_previo'] > u).values,
              '«siempre sube»': np.ones(len(df), bool)}
    filas = ''
    for nombre, pred in reglas.items():
        vp, fp, fn, prec, sens, f1 = metricas(real, pred)
        filas += f'<tr><td>{nombre}</td><td>{vp}</td><td>{fp}</td><td>{fn}</td><td>{prec:.3f}</td><td>{sens:.3f}</td><td>{f1:.3f}</td></tr>'
    txt13.value = (f'<b>{cj}</b>: {len(df):,} ejemplos · el río sube más de {u} cm en {h} h en el <b>{100 * real.mean():.1f} %</b> de los casos<br>'
                   f'Error de la regla «el nivel seguirá igual»: <b>{df["cambio"].abs().mean():.2f} cm</b>'.replace(',', ' ') +
                   '<table border="1" style="border-collapse:collapse;margin-top:6px;font-size:13px" cellpadding="4">'
                   '<tr><th>regla</th><th>VP</th><th>FP</th><th>FN</th><th>precisión</th><th>sensibilidad</th><th>F1</th></tr>' + filas + '</table>')
    with fig13.batch_update():
        fig13.data[0].x = df['cambio'].clip(-40, 40).values
        fig13.update_layout(title=dict(text=f'Cambio de nivel a +{h} h · línea naranja = umbral de {u} cm'),
                            shapes=[dict(type='line', x0=u, x1=u, y0=0, y1=1, yref='paper', line=dict(color=NARANJA, width=3))])
    with fig13b.batch_update():
        fig13b.data[0].x = df['fecha_objetivo'].values; fig13b.data[0].y = df['nivel'].values

display(widgets.VBox([widgets.HBox([w_h, w_u, w_c]), txt13, fig13, fig13b]))
vivo([w_h, w_u, w_c], actualiza13)

**Para explicar:** mueva el umbral y el conjunto. En *prueba* sube más seguido que en *entrenamiento* (más crecidas): por eso los errores de prueba son mayores. Ninguna regla simple detecta bien los aumentos y avisa a tiempo, y eso es lo que el MLP de la Fase 1 tampoco logró superar de forma demostrable.

In [ ]:
# 1.4 MINI-MLP (se entrena solo al soltar el deslizador): sobreajuste y regularización
# Versión simplificada de la Fase 1: 6 predictores, sin Dropout.
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

COLS = ['nivel', 'nivel_12h', 'nivel_24h', 'nivel_48h', 'nivel_72h', 'cambio_previo']

def entrena_mlp(neuronas=16, l2=1e-4, epocas=150, parada_temprana=True, semilla=0):
    df = particion(TABLAS[12]); tr, va, te = df['entrenamiento'], df['validación'], df['prueba']
    esc = StandardScaler().fit(tr[COLS]); s_y = tr['cambio'].std()
    Xtr, Xva, Xte = (esc.transform(a[COLS]) for a in (tr, va, te))
    ytr, yva = tr['cambio'].values / s_y, va['cambio'].values / s_y
    red = MLPRegressor(hidden_layer_sizes=(neuronas,), alpha=l2, learning_rate_init=1e-3, random_state=semilla)
    curvas, mejor, mejor_ep, pesos, sin_mejora = [], np.inf, 0, None, 0
    for ep in range(1, epocas + 1):
        red.partial_fit(Xtr, ytr)
        e_tr = np.mean((red.predict(Xtr) - ytr) ** 2) * s_y ** 2
        e_va = np.mean((red.predict(Xva) - yva) ** 2) * s_y ** 2
        curvas.append((e_tr, e_va))
        if e_va < mejor:
            mejor, mejor_ep, sin_mejora = e_va, ep, 0
            pesos = ([c.copy() for c in red.coefs_], [b.copy() for b in red.intercepts_])
        else:
            sin_mejora += 1
        if parada_temprana and sin_mejora >= 15:
            break
    if parada_temprana and pesos is not None:                 # restaura la mejor época
        red.coefs_, red.intercepts_ = pesos
    pred = red.predict(Xte) * s_y
    return {'curvas': np.array(curvas), 'mejor_ep': mejor_ep, 'epocas': len(curvas),
            'mae_modelo': float(np.mean(np.abs(pred - te['cambio'].values))),
            'mae_persistencia': float(te['cambio'].abs().mean())}

w_n = widgets.IntSlider(value=16, min=2, max=128, description='Neuronas')
w_l = widgets.FloatLogSlider(value=1e-4, base=10, min=-6, max=0, step=0.25, description='L2 (alpha)')
w_e = widgets.IntSlider(value=150, min=10, max=400, step=10, description='Épocas')
w_p = widgets.Checkbox(value=True, description='Parada temprana')
txt14 = widgets.HTML()
fig14 = go.FigureWidget(estilo(go.Figure(), 'Curvas de aprendizaje (MSE, escala logarítmica)', 340))
fig14.add_trace(go.Scatter(x=[0], y=[1], mode='lines', name='entrenamiento', line=dict(color=AZUL, width=2)))
fig14.add_trace(go.Scatter(x=[0], y=[1], mode='lines', name='validación', line=dict(color=NARANJA, width=2)))
fig14.update_yaxes(type='log', title='MSE (cm²)'); fig14.update_xaxes(title='época')

def actualiza14():
    txt14.value = 'Entrenando…'
    r = entrena_mlp(w_n.value, w_l.value, w_e.value, w_p.value)
    ep = np.arange(1, len(r['curvas']) + 1)
    with fig14.batch_update():
        fig14.data[0].x = ep; fig14.data[0].y = r['curvas'][:, 0]
        fig14.data[1].x = ep; fig14.data[1].y = r['curvas'][:, 1]
        fig14.update_layout(shapes=[dict(type='line', x0=r['mejor_ep'], x1=r['mejor_ep'], y0=0, y1=1, yref='paper', line=dict(color='black', dash='dash'))])
    txt14.value = (f'Épocas entrenadas: <b>{r["epocas"]}</b> · mejor época (validación): <b>{r["mejor_ep"]}</b><br>'
                   f'Error en prueba: modelo <b>{r["mae_modelo"]:.2f} cm</b> frente a persistencia <b>{r["mae_persistencia"]:.2f} cm</b>')

display(widgets.VBox([widgets.HBox([w_n, w_l]), widgets.HBox([w_e, w_p]), txt14, fig14]))
vivo([w_n, w_l, w_e, w_p], actualiza14, rapido=False)     # entrena al soltar el deslizador

**Para explicar:** sin parada temprana y con muchas épocas, la curva de validación se separa de la de entrenamiento: eso es **sobreajuste**. Subir el L2 o activar la parada temprana lo contiene. Compare el error de prueba con el de la persistencia: en la Fase 1 el MLP quedó cerca de ella y no la superó de forma demostrable. *Simplificación:* aquí hay 6 predictores y sin Dropout; el experimento oficial (con Dropout) está en el cuaderno de las Fases 1 y 2.

## 2. Fase 2 — El radar Sentinel-1
Se usan los recortes de 224 × 224 × 3 (VV, VH y VH−VV en dB) guardados en la Fase 3.

In [ ]:
# 2.1 Carga de los recortes de radar
assert RUTA_NPZ, 'Falta Fase_3/centinela_recortes_sar_224.npz en su Drive.'
datos = np.load(RUTA_NPZ, allow_pickle=True)
IMG = datos['X']; CAMBIO = datos['cambio_24h_cm'].astype(float)
ETIQ = datos['etiqueta'] if 'etiqueta' in datos.files else datos['y']
FECHAS = pd.to_datetime(datos['fecha'])
orden = np.argsort(FECHAS.values)                       # orden cronológico
print('Recortes:', IMG.shape, '| periodo:', FECHAS.min().date(), '->', FECHAS.max().date())
print('Etiquetas:', pd.Series(ETIQ).value_counts().to_dict())

In [ ]:
# 2.2 VISOR de escenas (se actualiza solo): qué «ve» el radar
CANALES = ['VV (dB)', 'VH (dB)', 'VH − VV (dB)']
fig22 = go.FigureWidget(make_subplots(rows=1, cols=3, subplot_titles=CANALES, horizontal_spacing=0.03))
for c in range(3):
    fig22.add_trace(go.Heatmap(z=np.zeros((2, 2)), colorscale='gray', showscale=False), row=1, col=c + 1)
    fig22.update_yaxes(autorange='reversed', showticklabels=False, scaleanchor=f'x{c + 1 if c else ""}', row=1, col=c + 1)
    fig22.update_xaxes(showticklabels=False, row=1, col=c + 1)
estilo(fig22, '', 400)
txt22 = widgets.HTML()
w_i = widgets.IntSlider(value=0, min=0, max=len(orden) - 1, description='Escena', layout=widgets.Layout(width='650px'))

def actualiza22():
    i = orden[w_i.value]
    with fig22.batch_update():
        for c in range(3):
            im = IMG[i, c][::2, ::2]; lo, hi = np.nanpercentile(im, [2, 98])
            fig22.data[c].z = im; fig22.data[c].zmin = float(lo); fig22.data[c].zmax = float(hi)
    txt22.value = (f'Escena <b>{w_i.value + 1}/{len(orden)}</b> · {FECHAS[i].date()} · etiqueta: <b>{ETIQ[i]}</b> · '
                   f'cambio de nivel a +24 h: <b>{CAMBIO[i]:+.1f} cm</b>')

display(widgets.VBox([w_i, txt22, fig22]))
vivo([w_i], actualiza22)

**Para explicar:** las imágenes se parecen mucho entre una fecha y otra y entre las clases (baja / estable / sube). Por eso el radar no se usa como foto, sino resumido; y por eso la rama de CNN sobre imágenes sola no encontró señal.

In [ ]:
# 2.3 «¿AYUDA EL RADAR?» (se actualiza solo) — juguete didáctico: solo radar (6 resúmenes por escena) -> cambio a +24 h
from sklearn.linear_model import Ridge

RESUMEN = np.stack([IMG[:, c].reshape(len(IMG), -1).mean(1) for c in range(3)] +
                   [IMG[:, c].reshape(len(IMG), -1).std(1) for c in range(3)], axis=1)[orden]
Y = CAMBIO[orden]

def prueba_radar(alpha=1.0, barajar=False, semilla=0):
    n = len(Y); i1, i2 = int(0.70 * n), int(0.85 * n)           # cronológica 70/15/15
    X = RESUMEN.copy()
    if barajar:                                                    # control: mezcla las escenas (rompe la relación fecha-radar)
        X = X[np.random.default_rng(semilla).permutation(n)]
    mu, sd = X[:i1].mean(0), X[:i1].std(0) + 1e-9
    X = (X - mu) / sd
    m = Ridge(alpha=alpha).fit(X[:i1], Y[:i1])
    mae = lambda a, b: float(np.mean(np.abs(a - b)))
    return {'prueba_escenas': n - i2, 'mae_modelo': mae(m.predict(X[i2:]), Y[i2:]),
            'mae_cero': mae(np.zeros(n - i2), Y[i2:]), 'mae_mediana': mae(np.full(n - i2, np.median(Y[:i1])), Y[i2:])}

w_a = widgets.FloatLogSlider(value=1.0, base=10, min=-2, max=4, step=0.25, description='Ridge alpha')
w_b = widgets.Checkbox(value=False, description='Barajar radar (control)')
w_s = widgets.IntSlider(value=0, min=0, max=20, description='Semilla')
txt23 = widgets.HTML()
fig23 = go.FigureWidget(estilo(go.Figure(), 'Error en prueba (cm): menor es mejor', 300))
fig23.add_trace(go.Bar(x=['«cambio = 0»', 'mediana', 'modelo con radar'], y=[0, 0, 0], marker_color=[GRIS, GRIS, AZUL], text=['', '', ''], textposition='outside'))
fig23.update_yaxes(title='MAE (cm)')

def actualiza23():
    r = prueba_radar(w_a.value, w_b.value, w_s.value)
    vals = [r['mae_cero'], r['mae_mediana'], r['mae_modelo']]
    with fig23.batch_update():
        fig23.data[0].y = vals; fig23.data[0].text = [f'{v:.2f}' for v in vals]
        fig23.data[0].x = ['«cambio = 0»', 'mediana de entrenamiento', 'radar BARAJADO' if w_b.value else 'modelo con radar']
        fig23.data[0].marker.color = [GRIS, GRIS, NARANJA if w_b.value else AZUL]
        fig23.update_yaxes(range=[0, max(vals) * 1.25])
    txt23.value = f'Escenas de prueba: <b>{r["prueba_escenas"]}</b>. Con pocas escenas, diferencias de décimas de cm son ruido.'

display(widgets.VBox([widgets.HBox([w_a, w_b, w_s]), txt23, fig23]))
vivo([w_a, w_b, w_s], actualiza23)

**Para explicar:** mire si el modelo con *solo radar y resúmenes simples* supera a «cambio = 0»: en la Rama A de la Fase 2 las imágenes solas no lo lograron (sin evidencia). El hallazgo positivo de la Fase 2 viene de **combinar la serie del río con 20 valores de radar por distancia al cauce** y validar por bloques de tiempo (13,70 → 11,54 cm a +24 h; 21,02 → 17,78 cm a +36 h, 5 de 5 bloques). Esos resultados oficiales están en el cuaderno de las Fases 1 y 2; esta celda es solo una demostración y el control «barajar» muestra cómo se verifica que un resultado no sea casualidad.

## 3. Fase 3 — GPU, precisión mixta y modelos para el borde

In [ ]:
# 3.1 BANCO DE PRUEBAS EN VIVO (se mide solo al cambiar el lote): float32 frente a float16 (AMP) con ResNet-18 en esta GPU
try:
    import torch, torchvision
    HAY_TORCH = True
except ImportError:
    HAY_TORCH = False
    print('PyTorch no está instalado en este entorno; use Google Colab.')

def bench(modo, lote, rep):
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    amp = (modo == 'float16 (AMP)') and dev == 'cuda'
    torch.manual_seed(0)
    red = torchvision.models.resnet18(weights=None, num_classes=3).to(dev).train()
    opt = torch.optim.Adam(red.parameters(), 1e-3); crit = torch.nn.CrossEntropyLoss()
    x = torch.randn(lote, 3, 224, 224, device=dev); y = torch.randint(0, 3, (lote,), device=dev)
    escala = torch.amp.GradScaler('cuda', enabled=amp) if dev == 'cuda' else None
    def paso():
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=dev, dtype=torch.float16, enabled=amp):
            perdida = crit(red(x), y)
        if escala is not None and amp:
            escala.scale(perdida).backward(); escala.step(opt); escala.update()
        else:
            perdida.backward(); opt.step()
    for _ in range(3): paso()                                    # calentamiento
    if dev == 'cuda':
        torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    for _ in range(rep): paso()
    if dev == 'cuda': torch.cuda.synchronize()
    ms = (time.time() - t0) / rep * 1000
    vram = torch.cuda.max_memory_allocated() / 1e6 if dev == 'cuda' else float('nan')
    del red, opt, x, y
    if dev == 'cuda': torch.cuda.empty_cache()
    return ms, lote / ms * 1000, vram

if HAY_TORCH:
    w_lote = widgets.Dropdown(options=[16, 32, 64, 128, 256, 512, 1024], value=32, description='Lote')
    w_rep = widgets.IntSlider(value=8, min=3, max=30, description='Repeticiones')
    txt31 = widgets.HTML()
    fig31 = go.FigureWidget(make_subplots(rows=1, cols=2, subplot_titles=('Milisegundos por lote (menos es mejor)', 'VRAM pico (MB)')))
    for k in range(2):
        fig31.add_trace(go.Bar(x=['float32', 'float16 (AMP)'], y=[0, 0], marker_color=[GRIS, VERDE], text=['', ''], textposition='outside'), row=1, col=k + 1)
    estilo(fig31, '', 340, showlegend=False)

    def actualiza31():
        dev = 'cuda' if torch.cuda.is_available() else 'cpu'
        txt31.value = 'Midiendo…'
        res = []
        for modo in ('float32', 'float16 (AMP)'):
            try:
                res.append(bench(modo, w_lote.value, w_rep.value))
            except RuntimeError as e:
                res.append(None)
                if dev == 'cuda': torch.cuda.empty_cache()
        etiquetas = ['float32', 'float16 (AMP)']
        with fig31.batch_update():
            for k, idx in enumerate((0, 2)):
                vals = [(r[idx] if r else 0) for r in res]
                fig31.data[k].y = vals
                fig31.data[k].text = [('CUDA OOM' if r is None else f'{v:.0f}') for r, v in zip(res, vals)]
        nombre = torch.cuda.get_device_name(0) if dev == 'cuda' else 'CPU (sin GPU: float16 no se usa)'
        extra = ''
        if res[0] and res[1]:
            extra = f' · aceleración float32 / float16: <b>{res[0][0] / res[1][0]:.2f}×</b>'
        txt31.value = f'Dispositivo: <b>{nombre}</b> · lote {w_lote.value}{extra}'

    display(widgets.VBox([widgets.HBox([w_lote, w_rep]), txt31, fig31]))
    vivo([w_lote, w_rep], actualiza31, rapido=False)

**Para explicar:** con lote 32 y 128 en la T4 el informe midió 1,97× y 2,09× al aislar el cómputo. Pruebe lote 1024: en float32 aparece **CUDA OOM** (se acaba la memoria) y en float16 todavía cabe. *Nota:* aquí se usan datos aleatorios y una ResNet-18 sin pesos preentrenados; mide velocidad y memoria, no calidad.

In [ ]:
# 3.2 Modelos exportados (lee los archivos reales de su Drive) y tabla antes/después de la compresión
carpeta_modelos = os.path.join(BASE, 'Fase_3', 'modelos')
archivos = sorted(glob.glob(os.path.join(carpeta_modelos, '*')))
print('Archivos en', carpeta_modelos)
for a in archivos:
    print(f'  {os.path.getsize(a) / 1e6:8.1f} MB  {os.path.basename(a)}')
if not archivos:
    print('  (no hay archivos: la tabla de abajo trae los valores medidos en la Fase 3)')

TABLA_F3 = pd.DataFrame([
    ['PyTorch float32 (.pth)', 44.8, 118.5, 0.293, 0.197, '0/11', '100 %'],
    ['ONNX float32 (.onnx)',   44.8,  35.3, 0.293, 0.197, '0/11', '100 %'],
    ['ONNX INT8 (.onnx)',      11.3,  23.8, 0.244, 0.131, '0/11', '95,1 %']],
    columns=['versión', 'tamaño (MB)', 'latencia mediana (ms)', 'exactitud prueba', 'F1 macro', 'aciertos «sube»', 'coincide con PyTorch'])
print('\nValores medidos en la Fase 3 (CPU de Colab, 1 hilo, lote 1; no es una tablet):')
print(TABLA_F3.to_string(index=False))

In [ ]:
# 3.3 ¿SIRVE COMO ALERTA? (se actualiza solo) Un umbral de calidad frente al modelo de imagen de la Fase 3
w_x = widgets.FloatSlider(value=0.60, min=0.0, max=1.0, step=0.05, description='Exigida')
txt33 = widgets.HTML()
fig33 = go.FigureWidget(estilo(go.Figure(), 'Exactitud en prueba', 260))
fig33.add_trace(go.Bar(y=['modelo', 'referencia «siempre baja»', 'exigida'], x=[0.293, 0.439, 0.6], orientation='h',
                       marker_color=[AZUL, GRIS, NARANJA], text=['0.293', '0.439', ''], textposition='outside'))
fig33.update_xaxes(range=[0, 1.05])

def actualiza33():
    ex = w_x.value; modelo, ref = 0.293, 0.439
    with fig33.batch_update():
        fig33.data[0].x = [modelo, ref, ex]; fig33.data[0].text = [f'{modelo:.3f}', f'{ref:.3f}', f'{ex:.2f}']
    cumple = modelo >= ex
    txt33.value = (f'<b>{"El modelo CUMPLE el mínimo." if cumple else "El modelo NO cumple el mínimo exigido."}</b><br>'
                   'Además no supera a la referencia trivial y acierta <b>0 de 11</b> imágenes «sube» (la clase que importa).')

display(widgets.VBox([w_x, txt33, fig33]))
vivo([w_x], actualiza33)

**Para explicar:** la cadena técnica (GPU, precisión mixta, ONNX, INT8 de 44,8 a 11,3 MB y de 35,3 a 23,8 ms) funciona; el **modelo** no es apto como alerta (acierta 0 de 11 «sube»). La ruta de despliegue propuesta (imágenes sin conexión con LiteRT; serie por API en la nube) y la recomendación de no desplegar como alerta están en el informe de la Fase 3. LiteRT no se ejecutó.

---
**Resumen para la sustentación.** Fase 1: la historia del río no supera a «seguirá igual» y la formulación binaria no es adecuada. Fase 2: el radar ayuda a +24 y +36 h (−16 % y −15 % de error) con controles; las ramas A, B y C no dieron evidencia concluyente. Fase 3: el despliegue técnico funciona; el modelo de imagen no sirve como alerta. Todo es un ejercicio de clase con un cliente hipotético.